In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

patients = spark.table("healthcare_demo.bronze.patients")
referrals = spark.table("healthcare_demo.bronze.referrals")
appointments = spark.table("healthcare_demo.bronze.appointments")
capacity = spark.table("healthcare_demo.bronze.capacity")
documents = spark.table("healthcare_demo.bronze.referral_documents")

In [0]:
patients_clean = (
    patients
    .filter(F.col("patient_id").isNotNull())
    .dropDuplicates(["patient_id"])
    .withColumn("area_code", F.upper(F.trim(F.col("area_code"))))
)

display(patients_clean)



In [0]:
referrals_clean = (
    referrals
    .filter(F.col("referral_id").isNotNull())
    .withColumn("referral_date", F.to_date("referral_date"))
    .withColumn("specialty", F.initcap(F.trim(F.col("specialty"))))
    .withColumn("priority", F.initcap(F.trim(F.col("priority"))))
    .dropDuplicates(["referral_id"])
)

display(referrals_clean)

In [0]:
appointments_clean = (
    appointments
    .filter(F.col("appointment_id").isNotNull())
    .withColumn("appointment_date", F.to_date("appointment_date"))
    .withColumn("appointment_status", F.initcap(F.trim(F.col("appointment_status"))))
    .dropDuplicates(["appointment_id"])
)

display(appointments_clean)

In [0]:
capacity_clean = (
    capacity
    .filter(F.col("location_id").isNotNull())
    .withColumn("specialty", F.initcap(F.trim(F.col("specialty"))))
    .withColumn("available_slots", F.col("available_slots").cast("int"))
    .dropDuplicates(["location_id", "specialty"])
)

display(capacity_clean)

In [0]:

# Transform unstructured referral documents

documents_transformed = (
    documents
    .withColumn(
        "document_text_lower",
        F.lower(F.coalesce(F.col("document_text"), F.lit("")))
    )
    .withColumn(
        "clinical_symptoms",
        F.when(
            F.col("document_text_lower").contains("chest discomfort") |
            F.col("document_text_lower").contains("breathlessness") |
            F.col("document_text_lower").contains("shortness of breath") |
            F.col("document_text_lower").contains("heart palpitations"),
            "Cardiac symptoms"
        ).when(
            F.col("document_text_lower").contains("knee") |
            F.col("document_text_lower").contains("joint stiffness") |
            F.col("document_text_lower").contains("reduced mobility"),
            "Musculoskeletal symptoms"
        ).when(
            F.col("document_text_lower").contains("skin") |
            F.col("document_text_lower").contains("rash") |
            F.col("document_text_lower").contains("inflammation"),
            "Skin symptoms"
        ).otherwise("Other symptoms")
    )
    .withColumn(
        "document_priority",
        F.when(
            F.col("document_text_lower").contains("urgent") |
            F.col("document_text_lower").contains("emergency"),
            "Urgent"
        ).otherwise("Routine")
    )
    .withColumn(
        "document_specialty",
        F.when(
            F.col("clinical_symptoms") == "Cardiac symptoms",
            "Cardiology"
        ).when(
            F.col("clinical_symptoms") == "Musculoskeletal symptoms",
            "Orthopaedics"
        ).when(
            F.col("clinical_symptoms") == "Skin symptoms",
            "Dermatology"
        ).otherwise("Unknown")
    )
    .drop("document_text_lower")
)

display(documents_transformed)


In [0]:
# Remove duplicate ingestion metadata before joining
patients_clean = patients_clean.drop("_ingested_at")
referrals_clean = referrals_clean.drop("_ingested_at")
appointments_clean = appointments_clean.drop("_ingested_at")
capacity_clean = capacity_clean.drop("_ingested_at")

# Join patients and referrals
patient_referrals = patients_clean.join(
    referrals_clean,
    on="patient_id",
    how="inner"
)

# Join appointments and clinic capacity
silver_data = (
    patient_referrals
    .join(appointments_clean, on="referral_id", how="left")
    .join(
        capacity_clean,
        on=["location_id", "specialty"],
        how="left"
    )
    .join(
        documents_transformed.select(
            "referral_id",
            "clinical_symptoms",
            "document_priority",
            "document_specialty"
        ),
        on="referral_id",
        how="left"
    )
    .withColumn("_silver_processed_at", F.current_timestamp())
)

In [0]:
silver_data = silver_data.withColumn(
    "waiting_days",
    F.datediff(
        F.coalesce(F.col("appointment_date"), F.current_date()),
        F.col("referral_date")
    )
)

display(silver_data)

In [0]:
silver_data.write.format("delta").option("mergeSchema", "true") \
    .mode("overwrite").saveAsTable(
    "healthcare_demo.silver.patient_referral_details"
)

In [0]:
%sql
SELECT *
FROM healthcare_demo.silver.patient_referral_details
LIMIT 20;

In [0]:
print("Silver record count:", silver_data.count())

print(
    "Missing patient IDs:",
    silver_data.filter(F.col("patient_id").isNull()).count()
)

print(
    "Missing referral IDs:",
    silver_data.filter(F.col("referral_id").isNull()).count()
)

print(
    "Missing capacity records:",
    silver_data.filter(F.col("available_slots").isNull()).count()
)

In [0]:

from pyspark.sql import functions as F

# Read Silver table
silver_df = spark.table(
    "healthcare_demo.silver.patient_referral_details"
)

# Define data quality checks
total_records = silver_df.count()

duplicate_referrals = (
    silver_df
    .groupBy("referral_id")
    .count()
    .filter(F.col("referral_id").isNotNull() & (F.col("count") > 1))
    .count()
)

missing_patient_id = silver_df.filter(
    F.col("patient_id").isNull()
).count()

missing_referral_id = silver_df.filter(
    F.col("referral_id").isNull()
).count()

missing_specialty = silver_df.filter(
    F.col("specialty").isNull()
).count()

missing_priority = silver_df.filter(
    F.col("priority").isNull()
).count()

missing_referral_date = silver_df.filter(
    F.col("referral_date").isNull()
).count()

invalid_priority = silver_df.filter(
    ~F.col("priority").isin("Urgent", "Routine") |
    F.col("priority").isNull()
).count()

invalid_specialty = silver_df.filter(
    ~F.col("specialty").isin(
        "Cardiology",
        "Dermatology",
        "Orthopaedics"
    ) |
    F.col("specialty").isNull()
).count()

negative_waiting_days = silver_df.filter(
    F.col("waiting_days") < 0
).count()

missing_waiting_days = silver_df.filter(
    F.col("waiting_days").isNull()
).count()

invalid_appointment_date = silver_df.filter(
    F.col("appointment_date").isNotNull() &
    F.col("referral_date").isNotNull() &
    (F.col("appointment_date") < F.col("referral_date"))
).count()

# Create data quality results
quality_results = [
    ("Total Silver records", total_records, "INFO"),
    ("Duplicate referral IDs", duplicate_referrals, "FAIL"),
    ("Missing patient IDs", missing_patient_id, "FAIL"),
    ("Missing referral IDs", missing_referral_id, "FAIL"),
    ("Missing specialties", missing_specialty, "FAIL"),
    ("Missing priorities", missing_priority, "FAIL"),
    ("Missing referral dates", missing_referral_date, "FAIL"),
    ("Invalid priorities", invalid_priority, "FAIL"),
    ("Invalid specialties", invalid_specialty, "FAIL"),
    ("Negative waiting days", negative_waiting_days, "FAIL"),
    ("Missing waiting days", missing_waiting_days, "FAIL"),
    ("Appointment before referral", invalid_appointment_date, "FAIL")
]

quality_df = spark.createDataFrame(
    quality_results,
    ["quality_check", "failed_records", "check_type"]
)

quality_df = quality_df.withColumn(
    "result",
    F.when(
        (F.col("check_type") == "INFO") |
        (F.col("failed_records") == 0),
        "PASS"
    ).otherwise("FAIL")
)

display(quality_df.orderBy("check_type", "quality_check"))

# Overall data quality status
failed_checks = quality_df.filter(
    (F.col("check_type") == "FAIL") &
    (F.col("result") == "FAIL")
).count()

if failed_checks == 0:
    print("Silver data quality validation PASSED")
else:
    print(f"Silver data quality validation FAILED: {failed_checks} checks need attention")


In [0]:
silver = spark.table(
    "healthcare_demo.silver.patient_referral_details"
)

display(
    silver.groupBy(
        "clinical_symptoms",
        "document_priority",
        "document_specialty"
    ).count().orderBy("document_specialty")
)